In [51]:

!pip install fpdf nltk -q

37002.41s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


You should consider upgrading via the '/Users/binitapandey/phishing-detection-ml-system/venv/bin/python3 -m pip install --upgrade pip' command.


In [52]:

import pandas as pd
import numpy as np
import re
import nltk
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
import os
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("All libraries loaded!")

All libraries loaded!


In [53]:

DATASET_PATH = '/content/drive/MyDrive/8th sem major project/sms_scam_detection_dataset_merged_with_lang1.csv'

df = pd.read_csv(DATASET_PATH)
print(f" Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

In [17]:

def clean_text(text):
    """
    Clean SMS text - ONLY clean the message, DO NOT replace URL, Email, Phone
    Keep all original content, just make it readable
    """
    if pd.isna(text):
        return ""

    text = str(text)

    abbreviations = {
        'u': 'you', 'r': 'are', 'ur': 'your', 'urs': 'yours',
        'd': 'the', 'y': 'why', 'b': 'be', 'c': 'see', 'k': 'okay',
        'n': 'and', 'w': 'with', 'w/o': 'without', 'w/': 'with',
        '4': 'for', '2': 'to', '2day': 'today', '2moro': 'tomorrow',
        '2nite': 'tonight', 'b4': 'before', 'pls': 'please',
        'plz': 'please', 'thx': 'thanks', 'ty': 'thank you',
        'lol': 'laugh out loud', 'brb': 'be right back',
        'btw': 'by the way', 'idk': 'i do not know',
        'imo': 'in my opinion', 'irl': 'in real life',
        'jk': 'just kidding', 'lmk': 'let me know',
        'np': 'no problem', 'omg': 'oh my god',
        'gtg': 'got to go', 'tbh': 'to be honest',
        'dm': 'direct message', 'pm': 'private message',
        'asap': 'as soon as possible', 'afk': 'away from keyboard',
        'bday': 'birthday', 'cya': 'see you', 'fyi': 'for your information',
        'hmu': 'hit me up', 'ily': 'i love you',
        'lmao': 'laughing my ass off', 'rofl': 'rolling on floor laughing',
        'smh': 'shaking my head', 'ttyl': 'talk to you later',
        'wyd': 'what are you doing', 'wtf': 'what the fuck',
        'yk': 'you know', 'yw': 'you\'re welcome',
        'omw': 'on my way', 'brb': 'be right back',
        'gtg': 'got to go', 'cya': 'see you',
        'da': 'the', 'ya': 'you', 'yall': 'you all',
        'l8r': 'later', 'gr8': 'great', 'g2g': 'got to go',
        'bbl': 'be back later', 'bbs': 'be back soon',
        'msg': 'message', 'txt': 'text', 'fav': 'favorite',
        'sec': 'second', 'mins': 'minutes', 'hrs': 'hours',
        'ppl': 'people', 'bc': 'because', 'cuz': 'because',
        'tbh': 'to be honest', 'idc': 'i do not care',
        'ily': 'i love you', 'ilu': 'i love you',
    }

    for short, full in sorted(abbreviations.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(r'\b' + re.escape(short) + r'\b', full, text, flags=re.IGNORECASE)

    text = re.sub(r'\s+([!?.])', r'\1', text)
    text = re.sub(r'([!?.])\s+', r'\1 ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'([!?.])\1+', r'\1', text)

    text = re.sub(r'\.{2,}', '.', text)

    text = text.lower()

    return text

print("🔄 Cleaning messages...")
df['cleaned_message'] = df['text'].apply(clean_text)

df['label'] = df['label'].str.lower()

df_clean = df[df['cleaned_message'].str.len() > 2].copy()
df_clean = df_clean.dropna(subset=['cleaned_message'])
df_clean = df_clean.drop_duplicates(subset=['cleaned_message'])

print(f" Cleaning complete!")
print(f"Original: {len(df)} | Cleaned: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} rows")

print("\n📊 Sample of cleaned data:")
display(df_clean[['label', 'cleaned_message']].head(10))

print("\n" + "="*60)
print("🔍 CLEANING EXAMPLES (URL, Email, Phone KEPT AS IS):")
print("="*60)

test_texts = [
    "Call me on 9854260260",
    "Visit my website http://example.com",
    "Email me at test@gmail.com",
    "My number is +91-9876543210",
    "Check www.google.com for more info",
    "Contact 98765 43210 or 98765-43210",
    "PLZ call me 2day at 1234567890",
    "Visit https://www.example.com/page for details"
]

for test in test_texts:
    cleaned = clean_text(test)
    print(f"Original: {test}")
    print(f"Cleaned:  {cleaned}")
    print("-"*50)

🔄 Cleaning messages...
✅ Cleaning complete!
Original: 139495 | Cleaned: 9841
Removed: 129654 rows

📊 Sample of cleaned data:


,label,cleaned_message
0,ham,your opinion about me? 1. over to. jada 3. kus...
1,ham,what's up? do you want me to come online? if y...
2,ham,so you workin overtime nigpun?
3,ham,"also sir, i sent you an email about how to log..."
4,spam,please stay at home. to encourage the notion o...
5,spam,bankofamerica alert 137943. please follow http...
6,ham,sorry dude. dont know how i forgot. even after...
7,ham,i don't quite know what to do. i still can't g...
8,ham,ok lor. anyway i thk we cant get tickets now c...
9,ham,wat are you doing now?



🔍 CLEANING EXAMPLES (URL, Email, Phone KEPT AS IS):
Original: Call me on 9854260260
Cleaned:  call me on 9854260260
--------------------------------------------------
Original: Visit my website http://example.com
Cleaned:  visit my website http://example.com
--------------------------------------------------
Original: Email me at test@gmail.com
Cleaned:  email me at test@gmail.com
--------------------------------------------------
Original: My number is +91-9876543210
Cleaned:  my number is +91-9876543210
--------------------------------------------------
Original: Check www.google.com for more info
Cleaned:  check www.google.com for more info
--------------------------------------------------
Original: Contact 98765 43210 or 98765-43210
Cleaned:  contact 98765 43210 or 98765-43210
--------------------------------------------------
Original: PLZ call me 2day at 1234567890
Cleaned:  please call me today at 1234567890
--------------------------------------------------
Original: Visit ht

In [21]:

def clean_text_for_pdf(text):
    """Clean text for PDF - remove problematic Unicode characters"""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace('\ufffd', '')  
    text = text.replace('\u2018', "'") 
    text = text.replace('\u2019', "'")  
    text = text.replace('\u201c', '"')  
    text = text.replace('\u201d', '"')  
    text = text.replace('\u2013', '-')  
    text = text.replace('\u2014', '-')  
    text = text.replace('\u2026', '...') 
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text

def create_pdf_report(df_clean, original_df, filename='cleaning_report.pdf'):
    """
    Generate a comprehensive PDF report with all cleaning details
    """
    print("📄 Generating PDF report...")

    total_original = len(original_df)
    total_cleaned = len(df_clean)
    removed = total_original - total_cleaned
    removed_percent = (removed / total_original) * 100 if total_original > 0 else 0

    spam_count = len(df_clean[df_clean['label'] == 'spam'])
    ham_count = len(df_clean[df_clean['label'] == 'ham'])
    spam_percent = (spam_count / total_cleaned) * 100 if total_cleaned > 0 else 0
    ham_percent = (ham_count / total_cleaned) * 100 if total_cleaned > 0 else 0

    avg_words = df_clean['cleaned_message'].str.split().str.len().mean()
    avg_chars = df_clean['cleaned_message'].str.len().mean()
    max_words = df_clean['cleaned_message'].str.split().str.len().max()
    min_words = df_clean['cleaned_message'].str.split().str.len().min()

    has_url = df_clean['cleaned_message'].str.contains(r'https?://|www\.', case=False, na=False).sum()
    has_email = df_clean['cleaned_message'].str.contains(r'@', na=False).sum()
    has_phone = df_clean['cleaned_message'].str.contains(r'\d{3}[-.]?\d{3}[-.]?\d{4}', na=False).sum()

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Arial', 'B', 24)
    pdf.cell(0, 20, 'SMS Spam Detection', ln=True, align='C')
    pdf.set_font('Arial', 'B', 18)
    pdf.cell(0, 15, 'Data Cleaning Report', ln=True, align='C')
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 10, f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True, align='C')
    pdf.ln(15)

    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '1. Dataset Overview', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"Total Original Messages: {total_original:,}", ln=True)
    pdf.cell(0, 8, f"Total Cleaned Messages: {total_cleaned:,}", ln=True)
    pdf.cell(0, 8, f"Removed Messages: {removed:,} ({removed_percent:.1f}%)", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Label Distribution:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Spam: {spam_count:,} ({spam_percent:.1f}%)", ln=True)
    pdf.cell(0, 8, f"  Ham: {ham_count:,} ({ham_percent:.1f}%)", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Message Statistics:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Average Word Count: {avg_words:.2f}", ln=True)
    pdf.cell(0, 8, f"  Average Character Count: {avg_chars:.2f}", ln=True)
    pdf.cell(0, 8, f"  Maximum Words: {max_words}", ln=True)
    pdf.cell(0, 8, f"  Minimum Words: {min_words}", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Features Preserved:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Messages with URLs: {has_url:,}", ln=True)
    pdf.cell(0, 8, f"  Messages with Emails: {has_email:,}", ln=True)
    pdf.cell(0, 8, f"  Messages with Phone Numbers: {has_phone:,}", ln=True)
    pdf.ln(10)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '2. Data Cleaning Steps Performed', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    cleaning_steps = [
        {
            'step': 'Step 1: SMS Abbreviation Expansion',
            'desc': 'Expanded common SMS abbreviations (u->you, pls->please, etc.)'
        },
        {
            'step': 'Step 2: Punctuation Spacing Fix',
            'desc': 'Fixed spacing around punctuation marks for better readability'
        },
        {
            'step': 'Step 3: Extra Whitespace Removal',
            'desc': 'Removed multiple spaces and normalized whitespace'
        },
        {
            'step': 'Step 4: Repeated Punctuation Removal',
            'desc': 'Removed repeated punctuation marks (!!, ??, etc.)'
        },
        {
            'step': 'Step 5: Lowercase Conversion',
            'desc': 'Converted all text to lowercase for consistency'
        },
        {
            'step': 'Step 6: Empty Message Removal',
            'desc': f'Removed messages with no content or very short content ({removed:,} messages)'
        },
        {
            'step': 'Step 7: Duplicate Message Removal',
            'desc': 'Removed duplicate messages to avoid bias in training'
        },
        {
            'step': 'Step 8: URL Preservation',
            'desc': 'Kept URLs as is without modification (http, https, www)'
        },
        {
            'step': 'Step 9: Email Preservation',
            'desc': 'Kept email addresses as is without modification'
        },
        {
            'step': 'Step 10: Phone Number Preservation',
            'desc': 'Kept phone numbers as is without modification'
        }
    ]

    for item in cleaning_steps:
        pdf.set_font('Arial', 'B', 12)
        pdf.cell(0, 8, f"  {item['step']}", ln=True)
        pdf.set_font('Arial', '', 11)
        pdf.multi_cell(0, 6, f"      {item['desc']}")
        pdf.ln(2)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '3. Sample of Cleaned Data', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 10)
    sample = df_clean[['label', 'cleaned_message']].head(8)

    for idx, row in sample.iterrows():
        pdf.set_font('Arial', 'B', 11)
        pdf.cell(0, 7, f"Message {idx+1} - Label: {row['label'].upper()}", ln=True)
        pdf.set_font('Arial', '', 10)
        text = clean_text_for_pdf(row['cleaned_message'])
        if len(text) > 90:
            text = text[:90] + "..."
        pdf.multi_cell(0, 6, f"  {text}")
        pdf.cell(0, 4, "-" * 60, ln=True)
        pdf.ln(2)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '4. Output Files Generated', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'CSV File:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, "  File Name: cleaned_sms_dataset.csv", ln=True)
    pdf.cell(0, 8, "  Columns: 2", ln=True)
    pdf.cell(0, 8, "    - label: Class label (spam/ham)", ln=True)
    pdf.cell(0, 8, "    - cleaned_message: Preprocessed text message", ln=True)
    pdf.cell(0, 8, f"  Total Rows: {total_cleaned:,}", ln=True)
    pdf.cell(0, 8, "  File Location: /content/drive/MyDrive/8th sem major project/", ln=True)
    pdf.ln(10)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'PDF Report:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, "  File Name: cleaning_report.pdf", ln=True)
    pdf.cell(0, 8, "  Content: Complete cleaning report with statistics and steps", ln=True)
    pdf.ln(10)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '5. Ready for Machine Learning Training', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.multi_cell(0, 8, "The cleaned dataset is now ready for machine learning training with 2 columns: label and cleaned_message.")
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 8, "Python Code Example for Model Training:", ln=True)
    pdf.set_font('Arial', '', 9)

    code_lines = [
        "# Load and prepare data",
        "import pandas as pd",
        "from sklearn.model_selection import train_test_split",
        "from sklearn.feature_extraction.text import TfidfVectorizer",
        "from sklearn.ensemble import RandomForestClassifier",
        "from sklearn.metrics import accuracy_score, classification_report",
        "",
        "# Load cleaned data",
        "df = pd.read_csv('cleaned_sms_dataset.csv')",
        "X = df['cleaned_message']",
        "y = df['label']",
        "",
        "# Vectorize text",
        "vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3))",
        "X_vec = vectorizer.fit_transform(X)",
        "",
        "# Split data",
        "X_train, X_test, y_train, y_test = train_test_split(",
        "    X_vec, y, test_size=0.2, random_state=42, stratify=y",
        ")",
        "",
        "# Train model",
        "model = RandomForestClassifier(n_estimators=100, random_state=42)",
        "model.fit(X_train, y_train)",
        "",
        "# Evaluate",
        "y_pred = model.predict(X_test)",
        "print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')",
        "print(classification_report(y_test, y_pred))"
    ]

    for line in code_lines:
        pdf.cell(0, 5, line, ln=True)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '6. Summary of Cleaning Actions', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 11)
    pdf.cell(70, 8, "Action", border=1)
    pdf.cell(90, 8, "Details", border=1)
    pdf.cell(30, 8, "Count", border=1, ln=True)

    pdf.set_font('Arial', '', 10)
    summary_data = [
        ("Original Messages", f"{total_original:,}", ""),
        ("Cleaned Messages", f"{total_cleaned:,}", ""),
        ("Removed Messages", f"{removed:,}", f"{removed_percent:.1f}%"),
        ("Spam Messages", f"{spam_count:,}", f"{spam_percent:.1f}%"),
        ("Ham Messages", f"{ham_count:,}", f"{ham_percent:.1f}%"),
        ("URLs Preserved", f"{has_url:,}", ""),
        ("Emails Preserved", f"{has_email:,}", ""),
        ("Phones Preserved", f"{has_phone:,}", ""),
        ("Average Words", f"{avg_words:.2f}", ""),
        ("Average Characters", f"{avg_chars:.2f}", ""),
        ("Max Words", f"{max_words}", ""),
        ("Min Words", f"{min_words}", ""),
    ]

    for row in summary_data:
        pdf.cell(70, 7, str(row[0]), border=1)
        pdf.cell(90, 7, str(row[1]), border=1)
        pdf.cell(30, 7, str(row[2]), border=1, ln=True)

    pdf.ln(10)

    pdf.set_font('Arial', 'I', 10)
    pdf.cell(0, 10, 'Generated by SMS Detection System - Google Colab', ln=True, align='C')

    try:
        pdf.output('cleaning_report.pdf', 'F')
        print(f" PDF saved locally: cleaning_report.pdf")

        import shutil
        drive_path = '/content/drive/MyDrive/8th sem major project/cleaning_report.pdf'
        shutil.copy('cleaning_report.pdf', drive_path)
        print(f" PDF copied to Google Drive: {drive_path}")

        if os.path.exists('cleaning_report.pdf'):
            file_size = os.path.getsize('cleaning_report.pdf')
            print(f" File size: {file_size:,} bytes")

        return 'cleaning_report.pdf'

    except Exception as e:
        print(f" Error saving PDF: {e}")

        try:
            pdf.output('cleaning_report.pdf')
            print(f"✅ PDF saved locally (alternative method): cleaning_report.pdf")
            return 'cleaning_report.pdf'
        except:
            print(" Failed to save PDF")
            return None

pdf_path = create_pdf_report(final_df, df)

if pdf_path and os.path.exists(pdf_path):
    print(f"\n📄 PDF Report Details:")
    print(f"  Location: {pdf_path}")
    print(f"  Size: {os.path.getsize(pdf_path):,} bytes")
    print(f"\n PDF generated successfully!")
else:
    print("\n⚠️ PDF could not be generated. Please check the error messages above.")

    print("\n🔄 Trying to generate a simpler PDF...")
    try:
        from fpdf import FPDF
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font('Arial', 'B', 16)
        pdf.cell(0, 10, 'SMS Spam Detection - Cleaning Report', ln=True)
        pdf.set_font('Arial', '', 12)
        pdf.cell(0, 8, f'Total Messages: {len(final_df):,}', ln=True)
        pdf.cell(0, 8, f'Spam: {len(final_df[final_df["label"]=="spam"]):,}', ln=True)
        pdf.cell(0, 8, f'Ham: {len(final_df[final_df["label"]=="ham"]):,}', ln=True)
        pdf.cell(0, 8, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', ln=True)
        pdf.output('simple_report.pdf')
        print(" Simple PDF saved as: simple_report.pdf")

        import shutil
        drive_path = '/content/drive/MyDrive/8th sem major project/simple_report.pdf'
        shutil.copy('simple_report.pdf', drive_path)
        print(f" Simple PDF copied to Drive: {drive_path}")

        pdf_path = 'simple_report.pdf'

    except Exception as e2:
        print(f" Simple PDF also failed: {e2}")

📄 Generating PDF report...
✅ PDF saved locally: cleaning_report.pdf
✅ PDF copied to Google Drive: /content/drive/MyDrive/8th sem major project/cleaning_report.pdf
✅ File size: 6,020 bytes

📄 PDF Report Details:
  Location: cleaning_report.pdf
  Size: 6,020 bytes

✅ PDF generated successfully!


In [23]:

from google.colab import files

print("📥 Download files:")

files.download('cleaned_sms_dataset.csv')

if pdf_path and os.path.exists(pdf_path):
    files.download(pdf_path)
elif os.path.exists('cleaning_report.pdf'):
    files.download('cleaning_report.pdf')

print(" Files downloaded!")


📥 Download files:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Files downloaded!


In [ ]:

print("="*60)
print(" CLEANING COMPLETE")
print("="*60)
print(f" Dataset Statistics:")
print(f"  Total: {len(final_df):,} messages")
print(f"  Spam: {len(final_df[final_df['label']=='spam']):,}")
print(f"  Ham: {len(final_df[final_df['label']=='ham']):,}")
print(f"\n📁 Files Saved in Google Drive:")
print(f"  📄 /content/drive/MyDrive/8th sem major project/cleaned_sms_dataset.csv")
print(f"  📄 /content/drive/MyDrive/8th sem major project/cleaning_report.pdf")
print("="*60)

In [12]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_sms_dataset.csv")

print(df.head())
print(df.info())
print(df['label'].value_counts())

  label                                    cleaned_message
0   ham  your opinion about me? 1. over to. jada 3. kus...
1   ham  what's up? do you want me to come online? if y...
2   ham                      so you are working overtime ?
3   ham  also sir, i sent you an email about how to log...
4  spam  please stay at home. to encourage the notion o...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   label            9841 non-null   object
 1   cleaned_message  9841 non-null   object
dtypes: object(2)
memory usage: 153.9+ KB
None
label
ham     8304
spam    1537
Name: count, dtype: int64


In [34]:
from sklearn.model_selection import train_test_split

X = df["cleaned_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [36]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (7872, 5000)
Testing TF-IDF shape: (1969, 5000)


In [37]:
from sklearn.naive_bayes import MultinomialNB

naiveb_model = MultinomialNB()
naiveb_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [38]:
y_pred_naiveb = naiveb_model.predict(X_test_tfidf)

In [39]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
print("===== NaiveByes =====")
print("Accuracy:", accuracy_score(y_test, y_pred_naiveb))

print("\nClassification Report")
print(classification_report(y_test, y_pred_naiveb))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_naiveb))

===== NaiveByes =====
Accuracy: 0.9598781107160995

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.75      0.85       308
           1       0.96      1.00      0.98      1661

    accuracy                           0.96      1969
   macro avg       0.97      0.87      0.92      1969
weighted avg       0.96      0.96      0.96      1969


Confusion Matrix
[[ 231   77]
 [   2 1659]]


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logisticr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logisticr_model.fit(X_train_tfidf, y_train)

y_pred_logisticr = logisticr_model.predict(X_test_tfidf)

In [21]:

print("===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred_logisticr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_logisticr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logisticr))

===== Logistic Regression =====
Accuracy: 0.9593702386998476

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98      1661
        spam       0.99      0.75      0.85       308

    accuracy                           0.96      1969
   macro avg       0.97      0.87      0.91      1969
weighted avg       0.96      0.96      0.96      1969


Confusion Matrix:
[[1659    2]
 [  78  230]]


In [22]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluation
print("===== Linear SVM =====")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

===== Linear SVM =====
Accuracy: 0.9766378872524124

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      1661
        spam       0.98      0.87      0.92       308

    accuracy                           0.98      1969
   macro avg       0.98      0.93      0.95      1969
weighted avg       0.98      0.98      0.98      1969


Confusion Matrix:
[[1656    5]
 [  41  267]]


In [25]:
pip install xgboost

You should consider upgrading via the '/Users/binitapandey/phishing-detection-ml-system/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_sms_dataset.csv")

In [24]:
print(df["label"].head())
print(df["label"].unique())

0     ham
1     ham
2     ham
3     ham
4    spam
Name: label, dtype: object
['ham' 'spam']


In [25]:
df["label"] = df["label"].map({
    "ham": 1,
    "spam": 0
})

print(df["label"].unique())

[1 0]


In [27]:
print(df["label"].unique())
print(y_train.unique())

[1 0]
['ham' 'spam']


In [29]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss"
)

xgb_model.fit(X_train_tfidf, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [30]:
print(df["label"].head(10))

0    1
1    1
2    1
3    1
4    0
5    0
6    1
7    1
8    1
9    1
Name: label, dtype: int64


In [31]:
print(df["label"].unique())
print(y_train.unique())
print(y_test.unique())

[1 0]
[1 0]
[1 0]


In [32]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss"
)

xgb_model.fit(X_train_tfidf, y_train)

y_pred_xgb = xgb_model.predict(X_test_tfidf)

print("===== XGBoost =====")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

===== XGBoost =====
Accuracy: 0.8435754189944135

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.00      0.01       308
           1       0.84      1.00      0.92      1661

    accuracy                           0.84      1969
   macro avg       0.67      0.50      0.46      1969
weighted avg       0.79      0.84      0.77      1969


Confusion Matrix:
[[   1  307]
 [   1 1660]]


In [40]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_balanced = LinearSVC(
    random_state=42,
    class_weight="balanced"
)

svm_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = svm_balanced.predict(X_test_tfidf)

print("===== Linear SVM (Balanced) =====")
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

===== Linear SVM (Balanced) =====
Accuracy: 0.9680040629761301

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       308
           1       0.98      0.98      0.98      1661

    accuracy                           0.97      1969
   macro avg       0.94      0.94      0.94      1969
weighted avg       0.97      0.97      0.97      1969


Confusion Matrix:
[[ 275   33]
 [  30 1631]]


In [41]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

cnb = ComplementNB()

cnb.fit(X_train_tfidf, y_train)

y_pred = cnb.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9319451498222447
              precision    recall  f1-score   support

           0       0.72      0.91      0.81       308
           1       0.98      0.94      0.96      1661

    accuracy                           0.93      1969
   macro avg       0.85      0.92      0.88      1969
weighted avg       0.94      0.93      0.94      1969

[[ 281   27]
 [ 107 1554]]


In [42]:
import joblib

In [43]:
joblib.dump(
    svm_model,
    "../models/message/svm_message_model.pkl"
)

print("Linear SVM model saved successfully!")

Linear SVM model saved successfully!


In [44]:
joblib.dump(
    vectorizer,
    "../models/message/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer saved successfully!")

TF-IDF Vectorizer saved successfully!


In [45]:
loaded_model = joblib.load("../models/message/svm_message_model.pkl")

loaded_vectorizer = joblib.load("../models/message/tfidf_vectorizer.pkl")

print("Model loaded successfully!")
print("Vectorizer loaded successfully!")

Model loaded successfully!
Vectorizer loaded successfully!


In [46]:
sample_message = [
    "Congratulations! You have won a $1000 prize. Click here to claim now."
]

sample_vector = loaded_vectorizer.transform(sample_message)

prediction = loaded_model.predict(sample_vector)

print(prediction)

['spam']


In [47]:
label = "Legitimate" if prediction[0] == 1 else "Phishing"

print(label)

Phishing
